<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Filtering in Frequency Domain</b></h1>
</div>

## Context

Frequency-domain processing represents an image as a superposition of spatial frequencies. Correct filtering requires understanding the 2-D DFT, spectrum interpretation, transfer functions, reconstruction, ringing, periodic noise, and quantitative validation.


## Problem Statement

Develop a reproducible Fourier-domain workflow that explains image spectra and applies low-pass, high-pass, band, notch, and illumination-related filters while connecting every frequency-domain operation to its spatial effect.


## Inputs and Fixed Parameters

Use module-local datasets under `../data/`, NumPy FFT conventions with explicit centering, repository-relative paths, real-valued reconstructed images after inverse transforms, and `../outputs/figures/` for diagnostics.


## 1. Spatial Frequency

A sinusoidal brightness pattern can be written as:

$$
g(x)=A\sin(2\pi f x+\phi)
$$

where:

- $A$ = amplitude;
- $f$ = spatial frequency;
- $\phi$ = phase.

Low spatial frequency means intensity changes slowly across space.  
High spatial frequency means intensity changes rapidly.

**Important:** high frequency does not mean high brightness.


## 2. Sinusoids, Complex Numbers, and the DFT

The DFT of a 1-D signal is:

$$
X[k]
=
\sum_{n=0}^{N-1}
x[n]e^{-j2\pi kn/N}
$$

Inverse:

$$
x[n]
=
\frac{1}{N}
\sum_{k=0}^{N-1}
X[k]e^{j2\pi kn/N}
$$

Euler's identity:

$$
e^{j\theta}
=
\cos(\theta)+j\sin(\theta)
$$

For $X=a+jb$:

$$
|X|=\sqrt{a^2+b^2}
$$

and

$$
\phi=\operatorname{atan2}(b,a)
$$

Magnitude = frequency strength.  
Phase = spatial alignment.


## 3. The 2-D Fourier Transform for Images

For image $f(x,y)$:

$$
F(u,v)
=
\sum_{x=0}^{M-1}
\sum_{y=0}^{N-1}
f(x,y)
e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$

The FFT computes the DFT efficiently.

`fftshift` moves the zero-frequency component to the center:

- center → low frequencies;
- farther from center → high frequencies.

Before inverse FFT, undo the shift with `ifftshift`.


## 4. Reading a 2-D Spectrum

A useful orientation rule:

> Spatial stripes produce spectral energy perpendicular to the stripe direction.

Let's prove it visually.


## 5. Inverse FFT and Reconstruction

The Fourier transform is reversible if we keep all coefficients.


## 6. Magnitude vs Phase

Every Fourier coefficient can be written as:

$$
F(u,v)=|F(u,v)|e^{j\phi(u,v)}
$$

Magnitude tells us how strong a frequency is.  
Phase strongly controls spatial organization.


## 7. Frequency-Domain Filtering

Let $F$ be the image spectrum and $H$ the filter:

$$
G(u,v)=H(u,v)F(u,v)
$$

Then:

$$
g(x,y)=\mathcal{F}^{-1}\{G(u,v)\}
$$

Workflow:

1. FFT;
2. center with `fftshift`;
3. construct $H$;
4. multiply $H\cdot F$;
5. undo shift;
6. IFFT;
7. keep the real component.


## 8. Frequency Distance Grid

For circular filters:

$$
D(u,v)=
\sqrt{(u-u_0)^2+(v-v_0)^2}
$$


## 9. Ideal, Gaussian, and Butterworth Low-Pass Filters

### Ideal LPF

$$
H(u,v)=
\begin{cases}
1,&D(u,v)\le D_0\\
0,&D(u,v)>D_0
\end{cases}
$$

### Gaussian LPF

$$
H(u,v)
=
\exp\left(
-\frac{D(u,v)^2}{2D_0^2}
\right)
$$

### Butterworth LPF

$$
H(u,v)
=
\frac{1}
{1+\left(\frac{D(u,v)}{D_0}\right)^{2n}}
$$

Butterworth order $n$ controls transition steepness.


## 10. Ringing and the Gibbs Phenomenon

A hard spectral cutoff corresponds to an oscillatory spatial response:

> abrupt spectral boundary → spatial oscillations → halos near edges


## 11. High-Pass Filtering

For a normalized LPF:

$$
H_{HP}=1-H_{LP}
$$

High frequencies contain edges and fine detail, but can also contain noise.


## 12. High-Boost Sharpening

A pure high-pass result mainly contains detail.

For sharpening:

$$
g(x,y)=f(x,y)+k f_{HP}(x,y)
$$


## 13. Convolution Theorem

$$
f*h
\quad\Longleftrightarrow\quad
F\cdot H
$$

Spatial convolution corresponds to multiplication in the frequency domain.

### Circular vs Linear Convolution

A DFT assumes periodic extension.

Therefore direct FFT multiplication naturally performs **circular convolution**.  
For ordinary linear convolution, appropriate zero-padding is generally required.


## 14. Band-Pass and Band-Reject Filters

Band-pass keeps:

$$
D_1\le D(u,v)\le D_2
$$

Band-reject removes that interval.


## 15. Periodic Noise

Periodic interference is one of the strongest reasons to use the frequency domain.

Repeated interference often becomes isolated off-center peaks in the spectrum.


## 16. Spectral Peak Detection

The following detector is intentionally simple:

1. remove the central low-frequency area;
2. rank remaining coefficients;
3. keep strong points separated by a minimum distance.


## 17. Notch-Reject Filtering

A notch-reject filter suppresses a small neighborhood around selected unwanted frequencies.

Real images have conjugate-symmetric spectra, so corresponding symmetric frequencies must also be considered.


## 18. Moiré Removal

Moiré is a repeated interference pattern. It can often be easier to isolate in the Fourier domain than in the spatial domain.


## 19. Slowly Varying Illumination / Shading

A simple multiplicative model is:

$$
I(x,y)\approx R(x,y)L(x,y)
$$

where:

- $R$ = reflectance / useful structure;
- $L$ = slowly varying illumination.

Because illumination varies slowly, it is dominated by low frequencies.


## 20. Cutoff Sensitivity

For a low-pass filter:

- smaller cutoff → stronger smoothing;
- larger cutoff → more detail preserved.


## 21. Quantitative Checks

MSE:

$$
\mathrm{MSE}
=
\frac{1}{MN}
\sum_{x,y}
[f(x,y)-g(x,y)]^2
$$

PSNR:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

PSNR measures numerical fidelity to a reference. It is not a universal perceptual-quality metric.


## 22. Validation Checks

Implement and validate this frequency-domain processing stage.


## 23. Failure Modes and Diagnostic Signatures

The following implementation failures have distinct diagnostic signatures:

- raw FFT magnitude obscures weak spectral components because of extreme dynamic range;
- missing `fftshift` / `ifftshift` misaligns the designed transfer function with the spectrum;
- confusing luminance level with spatial frequency leads to incorrect filter interpretation;
- aggressive Ideal cutoffs introduce ringing in the spatial domain;
- high-pass filtering can amplify acquisition noise together with fine structure;
- arbitrary suppression of bright spectral peaks can remove valid periodic texture;
- ignoring conjugate symmetry can produce inconsistent notch designs for real-valued images;
- insufficient zero-padding causes circular-convolution artifacts;
- PSNR alone does not establish perceptual or task-level improvement.

These signatures are used as diagnostic evidence during filter design and result review.

## 24. Parameter Sensitivity and Controlled Experiments

Execute controlled studies of spatial-frequency content, filter cutoff, Butterworth order, ringing, notch selection, moiré suppression, illumination correction, and zero-padding. Each study must isolate one design variable and produce interpretable evidence.

## 25. Method Selection and Technical Discussion

Document the criteria used to choose among Ideal, Gaussian, Butterworth, high-pass, band, and notch filters. Decisions must be tied to spectral signatures, reconstruction artifacts, and quantitative checks rather than formula recall.

## 26. Integrated Frequency-Domain Workflow

Consolidate the complete FFT → transfer function → inverse FFT processing chain and identify where spectral diagnostics, filter design, reconstruction, and validation enter the workflow.

## Completion Criterion

The Implementation notebook must execute end-to-end, reconstruct images correctly, generate the required spectral/spatial diagnostics, and pass its numerical validation checks.
